# Validación cruzada agrupando por embeddings (MLP)

# Nested CV con splits Cx explícitos — cuatro escenarios de generalización

Pipeline de evaluación de regresión logística + embeddings AlphaFold siguiendo los **cuatro escenarios de Park & Marcotte (2012, *Nature Methods*)** definidos en el esquema de selección de negativos:

| Escenario | Descripción | Generalización evaluada |
|-----------|-------------|------------------------|
| **C1** | Efector y proteína ya vistos en train | Más optimista (baseline) |
| **C2E** | Efector nuevo, proteína vista | Generalización sobre efectores |
| **C2P** | Proteína nueva, efector visto | Generalización sobre proteínas |
| **C3** | Ambos nuevos | Más exigente — uso real en inferencia |

Los splits de C2E, C2P y C3 se cargan directamente desde `generate_cx_splits.py` (evitando recomputar y garantizando reproducibilidad). C1 usa `StratifiedKFold` estándar como baseline sin restricción de leakage.

Al final del notebook se entrena un **modelo final** con todos los datos etiquetados y se realiza **inferencia sobre los casos dudosos**, indicando el nivel de confianza de cada predicción (C1/C2E/C2P/C3) según si el modelo ha visto moléculas similares durante el entrenamiento.

In [1]:
import sys
import json
import time
import warnings
from collections import defaultdict
from pathlib import Path

import numpy as np
import pandas as pd

from sklearn.pipeline import make_pipeline
from imblearn.pipeline import make_pipeline as make_pipeline_imb
from imblearn.over_sampling import RandomOverSampler
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.utils.class_weight import compute_sample_weight
from sklearn.model_selection import (
    GridSearchCV, cross_validate,
    BaseCrossValidator, StratifiedKFold,
)

# Generador de splits Cx (debe estar en el mismo directorio o en PYTHONPATH)
sys.path.insert(0, str(Path("/home/jovyan/TFG").resolve()))
from generate_cx_splits import build_and_save_splits, load_splits

In [2]:
# ── Paths ─────────────────────────────────────────────────────────────────────
OUTPUT_DIR      = Path("/home/jovyan/TFG/output_Efectores_alphafold_all")
OUTPUT_RESULTS = Path("/home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results_MLP_subniveles_C1/")
# PATH_LABELED    = OUTPUT_DIR / "labeled_interactions"    # pares etiquetados (pos + neg)
# PATH_UNKNOWN    = OUTPUT_DIR / "unknown_interactions"    # pares dudosos para inferencia
PATH_METADATA   = OUTPUT_RESULTS / "metadata.csv"           # sample_name, effector_group, protein_group, label
SPLITS_DIR      = OUTPUT_RESULTS / "splits"                 # directorio donde se guardan/leen los splits Cx

# Tipo de embedding y pooling por defecto (se sobreescribe tras la búsqueda)
DEFAULT_EMB_TYPE = "single_embeddings"

Preparamos los Data Frames para el entrenamiento y la inferencia que se usarán a continuación.

In [3]:
df_total = pd.read_csv("/home/jovyan/TFG/Interacciones_EffectorProteina_LiteratureOnly_Ordenadas_NleG.csv")
# Añadimos los grupos de efectores y proteínas
effector_groups = pd.read_csv("/home/jovyan/TFG/CV_Kmeans/effector_groups_kmeans.csv")
mapping_ef_groups = effector_groups.set_index("Effector")["Kmeans_Group"]
df_total["Effector_Group"] = df_total["Effector"].map(mapping_ef_groups)
protein_groups = pd.read_csv("/home/jovyan/TFG/CV_Kmeans/protein_groups_kmeans.csv")
mapping_prot_groups = protein_groups.set_index("Protein")["Kmeans_Group"]
df_total["Protein_Group"] = df_total["Protein"].map(mapping_prot_groups)
# Añadimos además una columna Sample name con el prefijo de las carpetas generadas por AlphaFold3
df_total["Sample_name"] = df_total["Protein"] + "_" + df_total["Effector"]
df_total.head()

,Effector,Protein,ProteinGeneName,Shared_Connections,Shortest_Path,Is_Connected,Effector_Group,Protein_Group,Sample_name
0,EspL,O89110,Casp8,4,1.0,True,Cluster_0,Cluster_1,O89110_EspL
1,EspL,Q60855,Ripk1,3,1.0,True,Cluster_0,Cluster_4,Q60855_EspL
2,NleB,O89110,Casp8,2,1.0,True,Cluster_0,Cluster_1,O89110_NleB
3,NleA,Q8R4B8,Nlrp3,1,1.0,True,Cluster_5,Cluster_4,Q8R4B8_NleA
4,NleA,Q9D8T2,Gsdmd,1,1.0,True,Cluster_5,Cluster_1,Q9D8T2_NleA


In [4]:
# Modificamos el Data Frame para que tenga solo la información que nos interesa
df_total = df_total.drop(columns=['Protein', 'Shared_Connections', 'Shortest_Path'])
df_total.rename(columns={'ProteinGeneName': 'Protein'}, inplace=True)
df_total.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Sample_name
0,EspL,Casp8,True,Cluster_0,Cluster_1,O89110_EspL
1,EspL,Ripk1,True,Cluster_0,Cluster_4,Q60855_EspL
2,NleB,Casp8,True,Cluster_0,Cluster_1,O89110_NleB
3,NleA,Nlrp3,True,Cluster_5,Cluster_4,Q8R4B8_NleA
4,NleA,Gsdmd,True,Cluster_5,Cluster_1,Q9D8T2_NleA


#### Pulido de las carpetas de output

En el output de AlphaFold3 hay carpetas que no fue capaz de ejecutar, con lo que se quedaron a medias. Esas carpetas nos interesa quitarlas del entrenamiento y de la inferencia porque no podemos trabajar con esos datos.

Localizamos qué muestras son y las quitamos del análisis.

In [5]:
incomplete_samples = set()

for sample in OUTPUT_DIR.iterdir():
    if sample.is_dir() and ".ipynb_checkpoints" not in sample.name:
        if not (sample / "TERMS_OF_USE.md") in sample.iterdir():
            # Nos quedamos con el nombre limpio y lo guardamos en la lista
            parts = sample.name.split('_')
            clean_name = "_".join(parts[:2])
            incomplete_samples.add(clean_name)

len(incomplete_samples)
incomplete_samples

{'B2RWS6_EspN',
 'B2RWS6_NleL',
 'P26039_EspL',
 'P26039_EspN',
 'P26039_NleL',
 'P26039_Tir'}

In [6]:
# Quitamos todas esas parejas que no ha conseguido procesar de la lista de interacciones totales y nos olvidamos de ellas
df_total = df_total[~df_total["Sample_name"].isin(incomplete_samples)]
len(df_total)

5466

In [7]:
# Creamos un Data Frame para las interacciones que formarán parte del train
# Añadimos una columna de Sample name que coincidirá con el prefijo de la carpeta generada por AlphaFold3
df_labeled = pd.read_csv("Interacciones_Entrenamiento_CV_estricto_kmeans_con_EspS_NleK.csv")
uniprot_equivalences = pd.read_csv("/home/jovyan/TFG/Interacciones_EffectorProteina_LiteratureOnly_Ordenadas_NleG.csv", sep=",")
prot_id = pd.Series(uniprot_equivalences.Protein.values, index=uniprot_equivalences.ProteinGeneName).to_dict()
df_labeled["Protein_ID"] = df_labeled["Protein"].map(prot_id)
df_labeled["Sample_name"] = df_labeled["Protein_ID"] + "_" + df_labeled["Effector"]
# y nos quedamos con las mismas columnas que en el Data Frame total
df_labeled = df_labeled.drop(columns=['Pathways_Shared', 'Shared_Connections', 'Shortest_Path', 'Interaction_Score', 'Protein_ID'])
# Es necesario además que quitemos de df_labeled también las muestras incomplete_samples
# por la misma razón de antes
df_labeled = df_labeled[~df_labeled["Sample_name"].isin(incomplete_samples)]
df_labeled.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,EspM,Rhoc,True,Cluster_6,Cluster_3,1,Q62159_EspM
1,NleB,Ripk1,True,Cluster_0,Cluster_4,1,Q60855_NleB
2,NleB,Nfkb1,True,Cluster_0,Cluster_4,1,P25799_NleB
3,EspH,Rac2,True,Cluster_1,Cluster_3,1,Q05144_EspH
4,NleD,Mapkapk2,True,Cluster_6,Cluster_1,1,P49138_NleD


In [8]:
# Repetimos ahora con las interacciones desconocidas que se usarán en la inferencia
train_samples = set(df_labeled["Sample_name"])
df_unknown = df_total[~df_total["Sample_name"].isin(train_samples)]
df_unknown.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Sample_name
25,NleD,Mapk14,True,Cluster_6,Cluster_3,P47811_NleD
46,EspF,Casp3,True,Cluster_2,Cluster_3,O08668_EspF
47,EspF,Casp6,True,Cluster_2,Cluster_3,O08738_EspF
48,EspF,Casp7,True,Cluster_2,Cluster_3,O08669_EspF
82,Tir,Casp4,True,Cluster_2,Cluster_1,P70343_Tir


In [9]:
# Comprobamos que las longitudes son correctas
print(f"Parejas etiquetadas: {len(df_labeled)}")
print(f"Parejas cuya interacción se desconoce: {len(df_unknown)}")
print(f"Parejas totales: {len(df_total)}")
print(f"Suma de parejas etiquetadas y desconocidas: {len(df_labeled)} + {len(df_unknown)} = {len(df_labeled) + len(df_unknown)}")

Parejas etiquetadas: 962
Parejas cuya interacción se desconoce: 4504
Parejas totales: 5466
Suma de parejas etiquetadas y desconocidas: 962 + 4504 = 5466


In [10]:
# Por último añadimos una columna de etiquetas a las parejas que se van a usar en el entrenamiento para saber si son positivas o negativas
df_labeled["Label"] = df_labeled["Is_Connected"].astype(int)
df_unknown["Label"] = "-"
df_total["Label"] = df_total.apply(
    lambda x: int(x["Is_Connected"]) if x["Sample_name"] in train_samples else "-",
    axis=1
)

# df_total.to_csv("Interacciones_totales_CV.csv")
# df_labeled.to_csv("Interacciones_entrenamiento_relajado_CV.csv")
# df_unknown.to_csv("Interacciones_inferencia_relajado_CV.csv")

df_labeled.head()

,Effector,Protein,Is_Connected,Effector_Group,Protein_Group,Label,Sample_name
0,EspM,Rhoc,True,Cluster_6,Cluster_3,1,Q62159_EspM
1,NleB,Ripk1,True,Cluster_0,Cluster_4,1,Q60855_NleB
2,NleB,Nfkb1,True,Cluster_0,Cluster_4,1,P25799_NleB
3,EspH,Rac2,True,Cluster_1,Cluster_3,1,Q05144_EspH
4,NleD,Mapkapk2,True,Cluster_6,Cluster_1,1,P49138_NleD


# Funciones

### 1. Carga de embeddings

Funciones para cargar embeddings de AlphaFold en *chunks* y construir bloques
(X, y, sample_names) para el entrenamiento. La función `load_block` también devuelve
los `sample_names` necesarios para mapear a los splits Cx.

Funciones para cargar embeddings de AlphaFold de acuerdo a las muestras presentes en el Data Frame proporcionado.

In [11]:
def load_block_from_csv(input_dir, target_df, emb_type="single_embeddings", transforms=None):
    import numpy as np
    X, y, sample_names = [], [], []

    for _, row in target_df.iterrows():
        sample_id = str(row['Sample_name']).strip()

        # --- CAMBIO AQUÍ: Buscar carpetas que EMPIECEN por el ID ---
        # Esto encontrará "ID", "ID_0", "ID_1", etc.
        matching_folders = list(input_dir.glob(f"{sample_id}*"))
        # Filtrar para asegurarnos de que sean directorios
        matching_folders = [f for f in matching_folders if f.is_dir()]

        if matching_folders:
            # Tomamos la primera coincidencia encontrada
            folder_path = matching_folders[0]

            # Buscamos el .npz dentro
            npz_folder = folder_path / "seed-0_embeddings"
            npz_files = list(npz_folder.glob("*.npz")) if npz_folder.exists() else []

            if npz_files:
                try:
                    path = npz_files[0]
                    emb_data = np.load(path)[emb_type]

                    if transforms is None:
                        X.append(emb_data)
                    elif len(transforms) == 1:
                        X.append(transforms[0](emb_data))
                    else:
                        X.append([t(emb_data) for t in transforms])

                    sample_names.append(sample_id)
                    if 'Label' in target_df.columns:
                        try:
                            # Intentamos convertir la etiqueta a entero (para entrenamiento)
                            y.append(int(row['Label']))
                        except (ValueError, TypeError):
                            # Si es un guion "-" o NaN (modo inferencia), añadimos un valor nulo o None
                            y.append(None)

                except Exception as e:
                    print(f"❌ Error al cargar datos de {sample_id}: {e}")
        else:
            # print(f"❓ No se encontró carpeta para: {sample_id}")
            pass

    X_array = np.asarray(X)
    if X_array.ndim > 2 and transforms and len(transforms) > 1:
        X_array = np.swapaxes(X_array, 0, 1)

    print(f"✅ Éxito: Se cargaron {len(X)} muestras de las {len(target_df)} del Excel.")
    return X_array, (np.array(y) if y else None), sample_names

### 2. Splits Cx — generación y carga

`build_and_save_splits` (de `generate_cx_splits.py`) genera y serializa los splits
C3/C2E/C2P a disco. `load_splits` los recupera.

`PrecomputedSplitter` traduce los splits (listas de `sample_name`) a índices de array,
haciéndolos compatibles con `cross_validate` y `GridSearchCV` de scikit-learn.

In [12]:
class PrecomputedSplitter(BaseCrossValidator):
    """
    Splitter de scikit-learn que usa splits precalculados (de generate_cx_splits).

    Traduce listas de sample_name a índices de numpy para compatibilidad
    con GridSearchCV y el bucle externo manual de hpm_search_nested_cv.
    """

    def __init__(self, folds: dict, sample_names: list):
        self.folds = folds
        self.name2idx = {name: i for i, name in enumerate(sample_names)}

    def _resolve(self, names):
        """Convierte lista de sample_name a array de índices (ignorando ausentes)."""
        return np.array(
            [self.name2idx[n] for n in names if n in self.name2idx],
            dtype=int,
        )

    def _iter_test_indices(self, X=None, y=None, groups=None):
        for fold in self.folds.values():
            yield self._resolve(fold["test"])

    def split(self, X, y=None, groups=None):
        for fold in self.folds.values():
            train_idx = self._resolve(fold["train"])
            test_idx  = self._resolve(fold["test"])
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return len(self.folds)

    def _iter_test_masks(self, X=None, y=None, groups=None):
        n = X.shape[0] if X is not None else len(self.name2idx)
        for test_idx in self._iter_test_indices(X, y, groups):
            mask = np.zeros(n, dtype=bool)
            if len(test_idx):
                mask[test_idx] = True
            yield mask


# ── Resolver Splitter Externo desde Carpeta C1 ────────────────────────────────
def get_outer_splitter_c1_sublevels(sublevel_prefix: str, sample_names: list, splits_dir) -> PrecomputedSplitter:
    """
    Carga el archivo splits_C1 y filtra los folds que pertenecen al subnivel activo.
    """
    from generate_cx_splits import load_splits
    all_c1_folds = load_splits(str(splits_dir), scenario="C1")
    
    filtered_folds = {
        k: v for k, v in all_c1_folds.items() 
        if k.startswith(sublevel_prefix)
    }
    
    if not filtered_folds:
        raise ValueError(f"No se encontraron folds con el prefijo '{sublevel_prefix}' en splits_C1_meta.json")
        
    return PrecomputedSplitter(filtered_folds, sample_names)


def get_inner_groups(groups_eg: np.ndarray, groups_pg: np.ndarray, scenario: str) -> np.ndarray:
    """
    Construye el array de grupos para el bucle INTERNO (GridSearchCV) a partir
    de los grupos del subconjunto de train del fold externo.

    El inner loop debe respetar la misma restricción que el outer para no
    introducir leakage en la selección de hiperparámetros
    (Krstajic et al., 2014, J Cheminform; Varoquaux et al., 2017, NeuroImage).

    Mapeo por escenario:
      C1  → None           (StratifiedKFold, sin grupos).
      C2E → groups_eg      (leave-one-effector_group-out en el inner train).
      C2P → groups_pg      (leave-one-protein_group-out en el inner train).
      C3  → eg + '__' + pg (leave-one-(eg × pg)-out en el inner train).
    """
    if scenario == "C1":
        return None
    if scenario == "C2E":
        return groups_eg
    if scenario == "C2P":
        return groups_pg
    if scenario == "C3":
        return np.array([f"{eg}__{pg}" for eg, pg in zip(groups_eg, groups_pg)])
    raise ValueError(f"Escenario desconocido: {scenario}")


def get_inner_splitter(scenario: str, inner_groups: np.ndarray, inner_cv: int = 5):
    """
    Devuelve (splitter_inner, groups_para_fit) para el GridSearchCV interno.

    Para C2E/C2P/C3 se usa LeaveOneGroupOut con los grupos del subconjunto de
    train, garantizando que la búsqueda de hiperparámetros no ve combinaciones
    no vistas en el fold de train externo.

    Para C1 se usa StratifiedKFold sin grupos.
    """
    from sklearn.model_selection import LeaveOneGroupOut
    if scenario == "C1":
        return StratifiedKFold(n_splits=inner_cv, shuffle=True, random_state=42), None
    return LeaveOneGroupOut(), inner_groups


### 3. Verificación de splits

Antes de entrenar, se imprime el resumen de cada fold: tamaño de train/test,
clases presentes y (para C3) muestras excluidas. Los folds con una sola clase
producirán NaN en las métricas durante el CV; se contabilizan y se informa.

In [13]:
def verify_cx_splits(scenario: str, splitter, X, y, sample_names, splits_dir):
    """
    Comprueba que PrecomputedSplitter ha cargado exactamente las particiones
    generadas por generate_cx_splits, comparando contra los metadatos JSON.

    Solo imprime el resumen final y los folds que no cuadren (si los hay).
    El reporte completo fold a fold ya está en splits_report.txt.

    :returns: (n_valid, n_one_class, n_empty)
    """
    import json as _json
    from pathlib import Path as _Path

    # C1 no tiene metadatos Cx — solo contar folds válidos
    if scenario == "C1":
        n_valid = n_one_class = n_empty = 0
        for train_idx, test_idx in splitter.split(X, y):
            if len(test_idx) == 0:
                n_empty += 1
            elif len(set(y[test_idx])) < 2:
                n_one_class += 1
            else:
                n_valid += 1
        viab = "✅ viable" if n_valid >= 3 else "❌ inviable"
        print(f"  [{scenario}] {splitter.get_n_splits(X, y)} folds  "
              f"válidos={n_valid}  una clase={n_one_class}  vacíos={n_empty}  {viab}")
        return n_valid, n_one_class, n_empty

    # Cargar metadatos (fuente de verdad de generate_cx_splits)
    meta_path = _Path(splits_dir) / f"splits_{scenario}_meta.json"
    with open(meta_path) as f:
        meta = _json.load(f)

    fold_labels = list(splitter.folds.keys())
    n_valid = n_one_class = n_empty = mismatches = 0
    mismatch_lines = []

    for label, (train_idx, test_idx) in zip(fold_labels, splitter.split(X, y)):
        if len(test_idx) == 0:
            n_empty += 1
            continue

        pos = int(y[test_idx].sum())
        neg = int(len(test_idx) - pos)

        if pos == 0 or neg == 0:
            n_one_class += 1
        else:
            n_valid += 1

        m = meta.get(str(label), {})
        errors = []
        if m.get("n_test_pos") != pos:
            errors.append(f"pos: got {pos} expected {m.get('n_test_pos')}")
        if m.get("n_test_neg") != neg:
            errors.append(f"neg: got {neg} expected {m.get('n_test_neg')}")
        if m.get("n_train") != len(train_idx):
            errors.append(f"train: got {len(train_idx)} expected {m.get('n_train')}")
        if errors:
            mismatches += 1
            mismatch_lines.append(f"    ❌ fold [{label}]: {', '.join(errors)}")

    viab = "✅ viable" if n_valid >= 3 else "❌ inviable"
    cross = "✅ coincide con report_splits" if mismatches == 0             else f"❌ {mismatches} fold(s) NO coinciden con report_splits"
    print(f"  [{scenario}] {len(fold_labels)} folds  "
          f"válidos={n_valid}  una clase={n_one_class}  vacíos={n_empty}  "
          f"{viab}  |  {cross}")
    for line in mismatch_lines:
        print(line)

    return n_valid, n_one_class, n_empty


### 4. Nested Cross-Validation multi-escenario

`hpm_search_nested_cv` es la función central: bucle externo con el splitter del
escenario elegido y bucle interno con `StratifiedKFold` para la búsqueda de
hiperparámetros. Los folds vacíos o de una clase retornan NaN (gestionados por
`np.nanmean` en el análisis posterior).

`test_pooling_transforms_all_scenarios` itera sobre todos los tipos de embedding,
poolings y escenarios en un único paso, devolviendo un diccionario
`results[scenario][emb_name]`.

In [14]:
# ── Motor de Búsqueda de Hiperparámetros (Nested CV para MLP) ─────────────────
def hpm_search_nested_cv_sublevels(
    X, y,
    outer_splitter,
    sublevel_name: str,
    inner_cv: int = 5,
):
    """
    Nested CV optimizado para clasificadores MLP usando subniveles de C1.
    Aplica Media clásica para C1_C2E y Acumulación (Pooling) para C1_C2P y C1_C3.
    Includes all extended metrics from general scenarios.
    """
    from sklearn.model_selection import StratifiedKFold, GridSearchCV
    from sklearn.metrics import (
        balanced_accuracy_score, matthews_corrcoef, precision_score, 
        recall_score, f1_score, roc_auc_score, average_precision_score,
        precision_recall_curve
    )
    import numpy as np
    
    param_grid = [
        {
            "mlpclassifier__hidden_layer_sizes": [(256, 128, 64), (128, 64, 32), (512, 256, 128)],
            "mlpclassifier__alpha":              [0.0001, 0.001, 0.01],
            "mlpclassifier__learning_rate_init": [0.001, 0.0001],
        }
    ]

    # Pipeline idéntico al de tu diseño para MLP desbalanceado
    pipeline = make_pipeline_imb(
        StandardScaler(),
        RandomOverSampler(random_state=42),
        MLPClassifier(
            activation='relu',
            solver='adam',
            max_iter=500,
            early_stopping=False, # Evita encoger aún más sets de train pequeños
            random_state=42,
        )
    )

    metrics_keys = (
        "test_bal_accuracy_50", "test_mcc_50", "test_precision_50", "test_recall_50", "test_pr_gain_50", "test_f1g_50",
        "test_bal_accuracy_opt", "test_mcc_opt", "test_precision_opt", "test_recall_opt", "test_pr_gain_opt", "test_f1g_opt",
        "test_best_threshold", "test_roc_auc", "test_pr_auc", "test_lift", "test_auprg", "test_expected_f1g"
    )
    
    results = {k: [] for k in metrics_keys}
    results["estimator"] = []
    results["fold_details"] = []

    # Vectores para acumulación (Pooling)
    y_true_acumulado = []
    y_proba_acumulado = []
    
    fold_ids = list(outer_splitter.folds.keys()) if hasattr(outer_splitter, "folds") else [f"fold_{i}" for i in range(outer_splitter.get_n_splits(X, y))]
    is_pooling_scenario = sublevel_name in ["C1_C2P", "C1_C3"]

    for fold_id, (train_idx, test_idx) in zip(fold_ids, outer_splitter.split(X, y)):
        if len(test_idx) == 0:
            for k in metrics_keys: results[k].append(np.nan)
            results["estimator"].append(None)
            results["fold_details"].append({
                "fold_id": fold_id, "n_test": 0, "n_test_pos": 0, "n_test_neg": 0,
                "roc_auc": np.nan, "pr_auc": np.nan, "lift": np.nan, "auprg": np.nan, "expected_f1g": np.nan,
                "best_threshold": np.nan, "precision_50": np.nan, "recall_50": np.nan, "bal_accuracy_50": np.nan,
                "mcc_50": np.nan, "pr_gain_50": np.nan, "f1g_50": np.nan, "precision_opt": np.nan, "recall_opt": np.nan,
                "bal_accuracy_opt": np.nan, "mcc_opt": np.nan, "pr_gain_opt": np.nan, "f1g_opt": np.nan,
                "best_hidden_layer_sizes": "", "best_alpha": np.nan, "best_lr_init": np.nan, "note": "vacío"
            })
            continue

        y_test = y[test_idx]
        X_train, y_train = X[train_idx], y[train_idx]
        X_test = X[test_idx]

        # Inner loop: StratifiedKFold limpio sin riesgo de fuga por el filtrado iterativo previo
        inner_splitter = StratifiedKFold(n_splits=inner_cv, shuffle=True, random_state=42)

        grid = GridSearchCV(
            pipeline, param_grid,
            cv=inner_splitter,
            scoring="average_precision", 
            n_jobs=-1,
            error_score=np.nan,
        )
        grid.fit(X_train, y_train)

        try:
            proba = grid.predict_proba(X_test)[:, 1]
            pi = float(y_test.sum() / len(y_test)) if len(y_test) > 0 else 0.0

            if is_pooling_scenario:
                y_true_acumulado.extend(y_test.tolist())
                y_proba_acumulado.extend(proba.tolist())

            # 1. EVALUACIÓN CON UMBRAL FIJO (0.50)
            pred_50 = (proba >= 0.5).astype(int)
            
            if len(y_test) == 1: # N=1 (Típico de C1_C3)
                bal_acc_50 = 1.0 if pred_50[0] == y_test[0] else 0.0
                mcc_50 = np.nan
                prec_50 = 1.0 if (pred_50[0] == 1 and y_test[0] == 1) else (0.0 if pred_50[0] == 1 else np.nan)
                rec_50  = 1.0 if (pred_50[0] == 1 and y_test[0] == 1) else (0.0 if y_test[0] == 1 else np.nan)
                f1_50   = 1.0 if (pred_50[0] == 1 and y_test[0] == 1) else 0.0
            else:
                bal_acc_50 = balanced_accuracy_score(y_test, pred_50)
                mcc_50     = matthews_corrcoef(y_test, pred_50)
                prec_50    = precision_score(y_test, pred_50, zero_division=0)
                rec_50     = recall_score(y_test, pred_50, zero_division=0)
                f1_50      = f1_score(y_test, pred_50, zero_division=0)
            
            pr_gain_50 = (prec_50 - pi) / ((1.0 - pi) * prec_50) if (not np.isnan(prec_50) and prec_50 > pi) else 0.0
            f1g_50     = (f1_50 - pi) / ((1.0 - pi) * f1_50) if f1_50 > pi else 0.0

            # 2. BARRIDO DE UMBRALES LOCALES (Solo si no es Pooling)
            if len(y_test) <= 1 or is_pooling_scenario:
                best_thresh, bal_acc_opt, mcc_opt, prec_opt, rec_opt, pr_gain_opt, f1g_opt = 0.5, bal_acc_50, mcc_50, prec_50, rec_50, pr_gain_50, f1g_50
                roc, pr, lift, auprg, expected_f1g = np.nan, np.nan, np.nan, np.nan, np.nan
            else:
                thresholds = np.linspace(0.01, 0.99, 100)
                best_f1, best_thresh = -1.0, 0.5
                for t in thresholds:
                    current_f1 = f1_score(y_test, (proba >= t).astype(int), zero_division=0)
                    if current_f1 > best_f1:
                        best_f1, best_thresh = current_f1, t

                pred_opt = (proba >= best_thresh).astype(int)
                bal_acc_opt = balanced_accuracy_score(y_test, pred_opt)
                mcc_opt     = matthews_corrcoef(y_test, pred_opt)
                prec_opt    = precision_score(y_test, pred_opt, zero_division=0)
                rec_opt     = recall_score(y_test, pred_opt, zero_division=0)
                pr_gain_opt = (prec_opt - pi) / ((1.0 - pi) * prec_opt) if prec_opt > pi else 0.0
                f1g_opt     = (best_f1 - pi) / ((1.0 - pi) * best_f1) if best_f1 > pi else 0.0

                roc  = roc_auc_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
                pr   = average_precision_score(y_test, proba) if len(np.unique(y_test)) > 1 else np.nan
                lift = pr / pi if (pi > 0 and not np.isnan(pr)) else np.nan
                
                # Integración geométrica para curvas completas (Solo C1_C2E)
                precisions, recalls, _ = precision_recall_curve(y_test, proba)
                with np.errstate(divide='ignore', invalid='ignore'):
                    prec_g_curve = (precisions - pi) / ((1.0 - pi) * precisions)
                    rec_g_curve = (recalls - pi) / ((1.0 - pi) * recalls)
                prec_g_curve = np.clip(np.nan_to_num(prec_g_curve, nan=0.0), 0.0, 1.0)
                rec_g_curve = np.clip(np.nan_to_num(rec_g_curve, nan=0.0), 0.0, 1.0)
                sort_idx = np.argsort(rec_g_curve)
                auprg = float(np.trapz(prec_g_curve[sort_idx], rec_g_curve[sort_idx]))
                
                zero_rec_indices = np.where(rec_g_curve[sort_idx] == 0.0)[0]
                y0 = float(prec_g_curve[sort_idx][zero_rec_indices[-1]]) if len(zero_rec_indices) > 0 else 0.0
                if y0 == 1.0:
                    expected_f1g = auprg / 2.0 + 0.25
                else:
                    expected_f1g = float((auprg / 2.0 + 0.25 - (pi * (1.0 - y0**2)) / 4.0) / (1.0 - pi * (1.0 - y0)))

            # Almacenar métricas en el diccionario principal
            results["test_bal_accuracy_50"].append(bal_acc_50)
            results["test_mcc_50"].append(mcc_50)
            results["test_precision_50"].append(prec_50)
            results["test_recall_50"].append(rec_50)
            results["test_pr_gain_50"].append(pr_gain_50)
            results["test_f1g_50"].append(f1g_50)
            
            results["test_bal_accuracy_opt"].append(bal_acc_opt)
            results["test_mcc_opt"].append(mcc_opt)
            results["test_precision_opt"].append(prec_opt)
            results["test_recall_opt"].append(rec_opt)
            results["test_pr_gain_opt"].append(pr_gain_opt)
            results["test_f1g_opt"].append(f1g_opt)
            
            results["test_best_threshold"].append(best_thresh)
            results["test_roc_auc"].append(roc)
            results["test_pr_auc"].append(pr)
            results["test_lift"].append(lift)
            results["test_auprg"].append(auprg)
            results["test_expected_f1g"].append(expected_f1g)
            results["estimator"].append(grid)
            
            # --- Extracción limpia de parámetros MLP para el reporte ---
            bp = grid.best_params_ if hasattr(grid, "best_params_") else {}
            hls_key   = next((k for k in bp if k.endswith("__hidden_layer_sizes")), None)
            alpha_key = next((k for k in bp if k.endswith("__alpha")), None)
            lr_key    = next((k for k in bp if k.endswith("__learning_rate_init")), None)

            results["fold_details"].append({
                "fold_id": fold_id, "n_test": len(test_idx),
                "n_test_pos": int(y_test.sum()), "n_test_neg": int(len(test_idx) - y_test.sum()),
                "roc_auc": round(roc, 4) if not np.isnan(roc) else np.nan,
                "pr_auc":  round(pr, 4) if not np.isnan(pr) else np.nan,
                "lift": round(lift, 4) if not np.isnan(lift) else np.nan,
                "auprg": round(auprg, 4) if not np.isnan(auprg) else np.nan,
                "expected_f1g": round(expected_f1g, 4) if not np.isnan(expected_f1g) else np.nan,
                "best_threshold": round(best_thresh, 4),
                
                "precision_50": round(prec_50, 4) if not np.isnan(prec_50) else np.nan,
                "recall_50": round(rec_50, 4) if not np.isnan(rec_50) else np.nan,
                "bal_accuracy_50": round(bal_acc_50, 4),
                "mcc_50": round(mcc_50, 4) if not np.isnan(mcc_50) else np.nan,
                "pr_gain_50": round(pr_gain_50, 4),
                "f1g_50": round(f1g_50, 4),
                
                "precision_opt": round(prec_opt, 4) if not np.isnan(prec_opt) else np.nan,
                "recall_opt": round(rec_opt, 4) if not np.isnan(rec_opt) else np.nan,
                "bal_accuracy_opt": round(bal_acc_opt, 4),
                "mcc_opt": round(mcc_opt, 4) if not np.isnan(mcc_opt) else np.nan,
                "pr_gain_opt": round(pr_gain_opt, 4),
                "f1g_opt": round(f1g_opt, 4),
                
                "best_hidden_layer_sizes": str(bp.get(hls_key, "")) if hls_key else "",
                "best_alpha": bp.get(alpha_key, np.nan) if alpha_key else np.nan,
                "best_lr_init": bp.get(lr_key, np.nan) if lr_key else np.nan,
                "note": "ok"
            })

        except Exception as e:
            for k in metrics_keys: results[k].append(np.nan)
            results["estimator"].append(None)
            results["fold_details"].append({
                "fold_id": fold_id, "n_test": len(test_idx), "n_test_pos": int(y_test.sum()), "n_test_neg": int(len(test_idx) - y_test.sum()),
                "roc_auc": np.nan, "pr_auc": np.nan, "lift": np.nan, "auprg": np.nan, "expected_f1g": np.nan,
                "best_threshold": np.nan, "precision_50": np.nan, "recall_50": np.nan, "bal_accuracy_50": np.nan,
                "mcc_50": np.nan, "pr_gain_50": np.nan, "f1g_50": np.nan, "precision_opt": np.nan, "recall_opt": np.nan,
                "bal_accuracy_opt": np.nan, "mcc_opt": np.nan, "pr_gain_opt": np.nan, "f1g_opt": np.nan,
                "best_hidden_layer_sizes": "", "best_alpha": np.nan, "best_lr_init": np.nan, "note": f"error: {str(e)}"
            })

    # =========================================================================
    # ── LOGICA DE POOLING CONSOLIDADO (MLP para C1_C2P y C1_C3) ──────────────
    # =========================================================================
    if is_pooling_scenario and len(y_true_acumulado) > 0 and len(np.unique(y_true_acumulado)) > 1:
        y_true_arr = np.array(y_true_acumulado)
        y_proba_arr = np.array(y_proba_acumulado)
        pi_global = float(np.mean(y_true_arr))
        
        g_roc  = float(roc_auc_score(y_true_arr, y_proba_arr))
        g_pr   = float(average_precision_score(y_true_arr, y_proba_arr))
        g_lift = float(g_pr / pi_global) if pi_global > 0 else np.nan
        
        # Integración geométrica global para el bloque acumulado completo
        precisions, recalls, _ = precision_recall_curve(y_true_arr, y_proba_arr)
        with np.errstate(divide='ignore', invalid='ignore'):
            prec_g_curve = (precisions - pi_global) / ((1.0 - pi_global) * precisions)
            rec_g_curve = (recalls - pi_global) / ((1.0 - pi_global) * recalls)
        prec_g_curve = np.clip(np.nan_to_num(prec_g_curve, nan=0.0), 0.0, 1.0)
        rec_g_curve = np.clip(np.nan_to_num(rec_g_curve, nan=0.0), 0.0, 1.0)
        sort_idx = np.argsort(rec_g_curve)
        g_auprg = float(np.trapz(prec_g_curve[sort_idx], rec_g_curve[sort_idx]))
        
        zero_rec_indices = np.where(rec_g_curve[sort_idx] == 0.0)[0]
        y0 = float(prec_g_curve[sort_idx][zero_rec_indices[-1]]) if len(zero_rec_indices) > 0 else 0.0
        g_f1g_e = float(g_auprg / 2.0 + 0.25) if y0 == 1.0 else float((g_auprg / 2.0 + 0.25 - (pi_global * (1.0 - y0**2)) / 4.0) / (1.0 - pi_global * (1.0 - y0)))

        # Búsqueda del umbral óptimo global sobre el pool (Fijo 0.5 ya se calculó fold a fold)
        thresholds = np.linspace(0.01, 0.99, 100)
        g_best_f1, g_best_thresh = -1.0, 0.5
        for t in thresholds:
            current_f1 = f1_score(y_true_arr, (y_proba_arr >= t).astype(int), zero_division=0)
            if current_f1 > g_best_f1:
                g_best_f1, g_best_thresh = current_f1, t
                
        pred_opt_g = (y_proba_arr >= g_best_thresh).astype(int)
        g_prec_opt = precision_score(y_true_arr, pred_opt_g, zero_division=0)
        g_rec_opt  = recall_score(y_true_arr, pred_opt_g, zero_division=0)
        g_bal_opt  = balanced_accuracy_score(y_true_arr, pred_opt_g)
        g_mcc_opt  = matthews_corrcoef(y_true_arr, pred_opt_g)
        g_prg_opt  = (g_prec_opt - pi_global) / ((1.0 - pi_global) * g_prec_opt) if g_prec_opt > pi_global else 0.0
        g_f1g_opt  = (g_best_f1 - pi_global) / ((1.0 - pi_global) * g_best_f1) if g_best_f1 > pi_global else 0.0

        # Cálculo de promedios para métricas estables de umbral fijo 0.50 globales en pooling
        g_bal_50 = np.nanmean(results["test_bal_accuracy_50"])
        g_mcc_50 = np.nanmean(results["test_mcc_50"])
        g_prec_50 = np.nanmean(results["test_precision_50"])
        g_rec_50 = np.nanmean(results["test_recall_50"])
        g_prg_50 = np.nanmean(results["test_pr_gain_50"])
        g_f1g_50 = np.nanmean(results["test_f1g_50"])
        
        # Inyección de los resultados fijos globales en cada fold del subnivel
        results["test_roc_auc"] = [g_roc] * len(fold_ids)
        results["test_pr_auc"] = [g_pr] * len(fold_ids)
        results["test_lift"] = [g_lift] * len(fold_ids)
        results["test_auprg"] = [g_auprg] * len(fold_ids)
        results["test_expected_f1g"] = [g_f1g_e] * len(fold_ids)
        results["test_best_threshold"] = [g_best_thresh] * len(fold_ids)
        
        results["test_bal_accuracy_50"] = [g_bal_50] * len(fold_ids)
        results["test_mcc_50"] = [g_mcc_50] * len(fold_ids)
        results["test_precision_50"] = [g_prec_50] * len(fold_ids)
        results["test_recall_50"] = [g_rec_50] * len(fold_ids)
        results["test_pr_gain_50"] = [g_prg_50] * len(fold_ids)
        results["test_f1g_50"] = [g_f1g_50] * len(fold_ids)
        
        results["test_bal_accuracy_opt"] = [g_bal_opt] * len(fold_ids)
        results["test_mcc_opt"] = [g_mcc_opt] * len(fold_ids)
        results["test_precision_opt"] = [g_prec_opt] * len(fold_ids)
        results["test_recall_opt"] = [g_rec_opt] * len(fold_ids)
        results["test_pr_gain_opt"] = [g_prg_opt] * len(fold_ids)
        results["test_f1g_opt"] = [g_f1g_opt] * len(fold_ids)
        
    return results



In [15]:
# ── Función de Control del Pipeline para Todos los Subniveles ─────────────────
def test_pooling_transforms_sublevels(transforms, labeled_dir, splits_dir, metadata_df: pd.DataFrame):
    """
    Orquesta la ejecución de la arquitectura MLP sobre los tres subniveles C1.
    Prints and tracks all specific threshold matrices.
    """
    import time
    import numpy as np
    from collections import defaultdict
    
    sublevels = {
        "C1_C2E": "C2E_",
        "C1_C2P": "C2P_",
        "C1_C3":  "C3_"
    }

    all_results = {}

    for sub_name, prefix in sublevels.items():
        print(f"\n{'#'*70}")
        print(f"#  SUBNIVEL INDIVIDUAL (MLP): {sub_name}")
        print(f"{'#'*70}")
        all_results[sub_name] = {}

        for emb_name, transform_list in transforms.items():
            print(f"\nEmbedding: {emb_name} ({len(transform_list)} poolings)")
            print("="*65)

            X_full, y_full, sample_names = load_block_from_csv(
                labeled_dir, 
                target_df=metadata_df[metadata_df['Label'].isin([0,1])], 
                emb_type=emb_name, 
                transforms=transform_list
            )

            if X_full.ndim == 3 and X_full.shape[0] != len(y_full):
                X_full = np.transpose(X_full, (1, 0, 2))

            res_emb = defaultdict(list)

            for i, _ in enumerate(transform_list):
                pooling_name = ["mean", "max", "min"][i] if i < 3 else f"pooling_{i}"
                print(f"\n  --- MLP | {emb_name} | {pooling_name} ---")

                Xi = X_full[:, i, :] if X_full.ndim == 3 else X_full
                outer_splitter = get_outer_splitter_c1_sublevels(prefix, sample_names, splits_dir)

                t0 = time.time()
                nested_res = hpm_search_nested_cv_sublevels(
                    Xi, y_full,
                    outer_splitter=outer_splitter,
                    sublevel_name=sub_name
                )
                elapsed = time.time() - t0

                # Captura estadística adaptativa extendida
                roc    = np.array(nested_res["test_roc_auc"], dtype=float)
                pr     = np.array(nested_res["test_pr_auc"], dtype=float)
                lift   = np.array(nested_res["test_lift"], dtype=float)
                auprg  = np.array(nested_res["test_auprg"], dtype=float)
                f1g_e  = np.array(nested_res["test_expected_f1g"], dtype=float)
                
                bal50  = np.array(nested_res["test_bal_accuracy_50"], dtype=float)
                mcc50  = np.array(nested_res["test_mcc_50"], dtype=float)
                pr50   = np.array(nested_res["test_precision_50"], dtype=float)
                rec50  = np.array(nested_res["test_recall_50"], dtype=float)
                prg50  = np.array(nested_res["test_pr_gain_50"], dtype=float)
                f50    = np.array(nested_res["test_f1g_50"], dtype=float)
                
                balopt = np.array(nested_res["test_bal_accuracy_opt"], dtype=float)
                mccopt = np.array(nested_res["test_mcc_opt"], dtype=float)
                propt  = np.array(nested_res["test_precision_opt"], dtype=float)
                rocopt = np.array(nested_res["test_recall_opt"], dtype=float)
                prgopt = np.array(nested_res["test_pr_gain_opt"], dtype=float)
                fopt   = np.array(nested_res["test_f1g_opt"], dtype=float)
                th     = np.array(nested_res["test_best_threshold"], dtype=float)

                n_valid_folds = int((~np.isnan(bal50)).sum())

                if sub_name in ["C1_C2P", "C1_C3"]:
                    mean_roc, mean_pr, mean_lift, mean_auprg, mean_f1g_e = roc[0], pr[0], lift[0], auprg[0], f1g_e[0]
                    mean_b50, mean_m50, mean_p50, mean_r50, mean_prg50, mean_f50 = bal50[0], mcc50[0], pr50[0], rec50[0], prg50[0], f50[0]
                    mean_bopt, mean_mopt, mean_popt, mean_ropt, mean_prgopt, mean_fopt, mean_th = balopt[0], mccopt[0], propt[0], rocopt[0], prgopt[0], fopt[0], th[0]
                    print(f"    [Métricas Consolidadas por Pooling de Bloque]")
                else:
                    mean_roc, mean_pr, mean_lift, mean_auprg, mean_f1g_e = np.nanmean(roc), np.nanmean(pr), np.nanmean(lift), np.nanmean(auprg), np.nanmean(f1g_e)
                    mean_b50, mean_m50, mean_p50, mean_r50, mean_prg50, mean_f50 = np.nanmean(bal50), np.nanmean(mcc50), np.nanmean(pr50), np.nanmean(rec50), np.nanmean(prg50), np.nanmean(f50)
                    mean_bopt, mean_mopt, mean_popt, mean_ropt, mean_prgopt, mean_fopt, mean_th = np.nanmean(balopt), np.nanmean(mccopt), np.nanmean(propt), np.nanmean(rocopt), np.nanmean(prgopt), np.nanmean(fopt), np.nanmean(th)
                    print(f"    [Media de los {len(roc)} Folds Evaluados]")

                print(f"    ROC AUC: {mean_roc:.4f} | PR AUC: {mean_pr:.4f} | Lift: {mean_lift:.2f}x")
                print(f"    AUPRG (PR-Gain): {mean_auprg:.4f} | E[FG1] (F1-Gain Esperado): {mean_f1g_e:.4f}")
                print(f"    [PUNTOS OPERATIVOS CONCRETOS]")
                print(f"      -> Umbral Fijo [0.50]:")
                print(f"         Prec: {mean_p50:.4f} | Rec: {mean_r50:.4f} | Bal.Acc: {mean_b50:.4f} | PR-Gain: {mean_prg50:.4f} | F1-Gain: {mean_f50:.4f}")
                print(f"      -> Umbral Optimizado [Promedio/Pool: {mean_th:.3f}]:")
                print(f"         Prec: {mean_popt:.4f} | Rec: {mean_ropt:.4f} | Bal.Acc: {mean_bopt:.4f} | PR-Gain: {mean_prgopt:.4f} | F1-Gain: {mean_fopt:.4f}")
                print(f"    Tiempo total de cómputo del subnivel: {elapsed:.1f}s")

                # Almacenamiento indexado completo
                res_emb["pooling_name"].append(pooling_name)
                res_emb["cv_roc_auc"].append(mean_roc)
                res_emb["cv_pr_auc"].append(mean_pr)
                res_emb["cv_lift"].append(mean_lift)
                res_emb["cv_auprg"].append(mean_auprg)
                res_emb["cv_expected_f1g"].append(mean_f1g_e)
                
                res_emb["cv_precision_50"].append(mean_p50)
                res_emb["cv_recall_50"].append(mean_r50)
                res_emb["cv_bal_accuracy_50"].append(mean_b50)
                res_emb["cv_mcc_50"].append(mean_m50)
                res_emb["cv_pr_gain_50"].append(mean_prg50)
                res_emb["cv_f1g_50"].append(mean_f50)
                
                res_emb["cv_best_threshold"].append(mean_th)
                res_emb["cv_precision_opt"].append(mean_popt)
                res_emb["cv_recall_opt"].append(mean_ropt)
                res_emb["cv_bal_accuracy_opt"].append(mean_bopt)
                res_emb["cv_mcc_opt"].append(mean_mopt)
                res_emb["cv_pr_gain_opt"].append(mean_prgopt)
                res_emb["cv_f1g_opt"].append(mean_fopt)
                
                res_emb["n_valid_folds"].append(n_valid_folds)
                res_emb["opt_time"].append(elapsed)
                res_emb["estimators"].append(nested_res["estimator"])
                res_emb["fold_details"].append(nested_res["fold_details"])

            all_results[sub_name][emb_name] = res_emb

    return all_results


### 5. Prueba de aleatoriedad

Verifica que el pipeline no aprende de artefactos estructurales: se asignan
etiquetas aleatorias y se espera accuracy ≈ 0.50. Se usa el escenario C1
(StratifiedKFold) para rapidez.

In [16]:
def sanity_check_random_labels_sublevels(X, y, n_repeats=3):
    """
    Entrena el MLP barajando las etiquetas al azar para confirmar la ausencia de leakage.
    Usa el escenario C1 de base para una comprobación rápida.
    """
    import numpy as np
    from sklearn.model_selection import StratifiedKFold
    
    print("\n" + "="*55)
    print(" === SANITY CHECK: ETIQUETAS ALEATORIAS (Escenario C1) ===")
    print("="*55)
    
    accs = []
    
    for i in range(n_repeats):
        y_rand = np.random.permutation(y)
        outer_c1 = StratifiedKFold(n_splits=5, shuffle=True, random_state=42 + i)
        
        res = hpm_search_nested_cv_sublevels(
            X=X, 
            y=y_rand, 
            outer_splitter=outer_c1,
            sublevel_name="C1",  # Usamos C1 para que el motor aplique la media clásica fold a fold
            inner_cv=5
        )
        
        acc = np.nanmean(res["test_bal_accuracy_50"])
        accs.append(acc)
        print(f"  Repetición {i+1}: balanced accuracy = {acc:.4f}")
        
    media_global = np.mean(accs)
    print("─" * 55)
    print(f"  Media total: {media_global:.4f}  (Esperado científico ≈ 0.50)")
    
    if media_global > 0.55:
        print("  ⚠️ ALERTA: Rendimiento inusualmente alto con datos al azar.")
        print("  Revisa si hay variables correlacionadas duplicadas o fugas en el preprocesamiento.")
    else:
        print("  ✅ ÉXITO: El modelo rinde al azar con etiquetas aleatorias. Sin leakage.")
    print("=" * 55)

### 6. Tabla de resultados por escenario

In [17]:
def optresults2table_nested(all_results):
    """
    Convierte el dict de resultados (todos los escenarios/subniveles) en un DataFrame.
    """
    import pandas as pd
    import numpy as np
    def get_mode(lst): 
        return max(set(lst), key=lst.count) if lst else np.nan
    
    rows = []
    for scenario, scenario_res in all_results.items():
        for emb_name, res in scenario_res.items():
            for i in range(len(res["pooling_name"])):

                estimators = [
                    e for e in res["estimators"][i]
                    if e is not None and hasattr(e, "best_params_")
                ]

                if estimators:
                    param_keys = list(estimators[0].best_params_.keys())
                    hls_k   = next((k for k in param_keys if k.endswith("__hidden_layer_sizes")), None)
                    alpha_k = next((k for k in param_keys if k.endswith("__alpha")), None)
                    lr_k    = next((k for k in param_keys if k.endswith("__learning_rate_init")), None)
                    
                    hls_list   = [str(e.best_params_[hls_k])   for e in estimators if hls_k]
                    alpha_list = [e.best_params_[alpha_k]       for e in estimators if alpha_k]
                    lr_list    = [e.best_params_[lr_k]          for e in estimators if lr_k]
                else:
                    hls_list = alpha_list = lr_list = []

                rows.append({
                    "scenario":         scenario,
                    "representation":   emb_name,
                    "pooling":          res["pooling_name"][i],
                    "roc_auc":          res["cv_roc_auc"][i],
                    "pr_auc":           res["cv_pr_auc"][i],
                    "pr_auc_lift":      res["cv_lift"][i],
                    "auprg":            res["cv_auprg"][i],
                    "expected_f1_gain": res["cv_expected_f1g"][i],
                    
                    # Umbral Fijo 0.50
                    "precision_50":     res["cv_precision_50"][i],
                    "recall_50":        res["cv_recall_50"][i],
                    "bal_accuracy_50":  res["cv_bal_accuracy_50"][i],
                    "mcc_50":           res["cv_mcc_50"][i],
                    "pr_gain_50":       res["cv_pr_gain_50"][i],
                    "f1_gain_50":       res["cv_f1g_50"][i],
                    
                    # Umbral Optimizado
                    "best_threshold":   res["cv_best_threshold"][i],
                    "precision_opt":    res["cv_precision_opt"][i],
                    "recall_opt":       res["cv_recall_opt"][i],
                    "bal_accuracy_opt": res["cv_bal_accuracy_opt"][i],
                    "mcc_opt":          res["cv_mcc_opt"][i],
                    "pr_gain_opt":      res["cv_pr_gain_opt"][i],
                    "f1_gain_opt":      res["cv_f1g_opt"][i],
                    
                    "time_s":           res["opt_time"][i],
                    "best_hidden_layer_sizes": get_mode(hls_list),
                    "best_alpha":               get_mode(alpha_list),
                    "best_lr_init":             get_mode(lr_list),
                    "n_valid_folds":            res["n_valid_folds"][i],
                })

    df = pd.DataFrame(rows)
    return df.sort_values(["scenario", "auprg"], ascending=[True, False]).reset_index(drop=True)

### 6b. Tabla de resultados por fold individual

Muestra qué grupo o combinación estaba en el test set de cada fold, su composición (n_pos, n_neg) y las métricas obtenidas. Especialmente útil cuando hay pocos folds (C3 con 4 folds) y la varianza entre folds es informativa por sí misma.

In [18]:
def fold_detail_table(all_results: dict) -> pd.DataFrame:
    """
    Genera una tabla con los resultados detallados por fold individual.
    """
    import pandas as pd
    import numpy as np
    
    rows = []
    for scenario, scenario_res in all_results.items():
        for emb_name, res in scenario_res.items():
            for i, pooling_name in enumerate(res["pooling_name"]):
                for fold_info in res["fold_details"][i]:
                    rows.append({
                        "scenario":       scenario,
                        "representation": emb_name,
                        "pooling":        pooling_name,
                        "fold_id":        fold_info["fold_id"],
                        "note":           fold_info["note"],
                        "n_test":         fold_info.get("n_test", 0),
                        "n_test_pos":     fold_info.get("n_test_pos", 0),
                        "roc_auc":        fold_info.get("roc_auc", np.nan),
                        "pr_auc":         fold_info.get("pr_auc", np.nan),
                        "lift":           fold_info.get("lift", np.nan),
                        "auprg":          fold_info.get("auprg", np.nan),
                        "expected_f1g":   fold_info.get("expected_f1g", np.nan),
                        
                        # Detalles 50
                        "precision_50":    fold_info.get("precision_50", np.nan),
                        "recall_50":       fold_info.get("recall_50", np.nan),
                        "bal_accuracy_50": fold_info.get("bal_accuracy_50", np.nan),
                        "mcc_50":          fold_info.get("mcc_50", np.nan),
                        "pr_gain_50":      fold_info.get("pr_gain_50", np.nan),
                        "f1g_50":          fold_info.get("f1g_50", np.nan),
                        
                        # Detalles Opt
                        "best_threshold":   fold_info.get("best_threshold", np.nan),
                        "precision_opt":    fold_info.get("precision_opt", np.nan),
                        "recall_opt":       fold_info.get("recall_opt", np.nan),
                        "bal_accuracy_opt": fold_info.get("bal_accuracy_opt", np.nan),
                        "mcc_opt":          fold_info.get("mcc_opt", np.nan),
                        "pr_gain_opt":      fold_info.get("pr_gain_opt", np.nan),
                        "f1g_opt":          fold_info.get("f1g_opt", np.nan),
                        
                        "best_hidden_layer_sizes": fold_info.get("best_hidden_layer_sizes", ""),
                        "best_alpha":              fold_info.get("best_alpha", np.nan),
                        "best_lr_init":            fold_info.get("best_lr_init", np.nan),
                    })
                    
    df = pd.DataFrame(rows)
    return df.sort_values(["scenario", "representation", "pooling", "fold_id"]).reset_index(drop=True)

# Ejecución principal

In [19]:
# ── 1. Definir transformaciones de pooling a comparar ────────────────────────
test_transforms = {
    "single_embeddings": [
        lambda x: np.mean(x, axis=0),
        lambda x: np.max(x, axis=0),
        lambda x: np.min(x, axis=0),
    ],
    "pair_embeddings": [
        lambda x: np.mean(x, axis=(0, 1)),
        lambda x: np.max(x, axis=(0, 1)),
        lambda x: np.min(x, axis=(0, 1)),
    ],
}

warnings.filterwarnings("ignore")

# Repetición validación cruzada añadiendo subniveles en C1

In [20]:
# Generación de los nuevos splits
# Generador de splits Cx con subniveles C1 (debe estar en el mismo directorio o en PYTHONPATH)
sys.path.insert(0, str(Path("/home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/").resolve()))
from generate_cx_splits_individual_sublevels import build_and_save_splits, load_splits

### Generación de splits

In [21]:
labeled_meta = df_labeled

# Generar (o regenerar) los splits exclusivos de subniveles C1
splits = build_and_save_splits(
    labeled_meta,
    output_dir=str("results/splits_subniveles"),
    effector_col="Effector",         # Nombre de tu columna individual
    protein_col="Protein",           # Nombre de tu columna individual
    label_col="Label",
    sample_col="Sample_name",
    min_train_ratio=0.50             # Mantiene el train por encima del 50% del tamaño original
)

print(f"\nDataset: {len(labeled_meta)} parejas  "
      f"({(labeled_meta.Label==1).sum()} pos, {(labeled_meta.Label==0).sum()} neg)")
print(f"Splits de C1 guardados en: {SPLITS_DIR}")

  REPORTE CV — SUBNIVELES EXHAUSTIVOS C1 CON FILTRO BICLASE ITERATIVO

Escenario C1 (1106 folds generados exitosamente)
───────────────────────────────────────────────────────────────────────────
  → Subnivel: C1_C2E (24 folds)
    Fold [C2E_eff_EspM                       ] train= 936 test=21 (pos=10, neg=11)
    Fold [C2E_eff_NleB                       ] train= 920 test=34 (pos=17, neg=17)
    Fold [C2E_eff_EspH                       ] train= 937 test=20 (pos=10, neg=10)
    Fold [C2E_eff_NleD                       ] train= 893 test=43 (pos=22, neg=21)
    Fold [C2E_eff_NleF                       ] train= 927 test=30 (pos=5, neg=25)
    [... y 19 folds más de este subnivel ...]
  → Subnivel: C1_C2P (120 folds)
    Fold [C2P_prot_Rhoc                      ] train= 951 test= 6 (pos=4, neg=2)
    Fold [C2P_prot_Ripk1                     ] train= 954 test= 3 (pos=2, neg=1)
    Fold [C2P_prot_Nfkb1                     ] train= 952 test= 5 (pos=3, neg=2)
    Fold [C2P_prot_Rac2             

  💾 Datos guardados con éxito en results/splits_subniveles/

Dataset: 962 parejas  (199 pos, 763 neg)
Splits de C1 guardados en: /home/jovyan/TFG/CV_Kmeans_con_EspS_NleK_all_models/results_MLP_subniveles_C1/splits


In [22]:
# Comprobamos que está asegurándose de que en el train de cada fold se cumple el nivel estricto

# 1. Cargar la matriz de roles que se acaba de guardar
# (Asegúrate de apuntar a la ruta correcta de tu SPLITS_DIR)
SPLITS_DIR_SUBNIVELES = "results/splits_subniveles/"
folds = load_splits(SPLITS_DIR_SUBNIVELES, "C1")

# 2. Elegimos un fold cualquiera para auditar, por ejemplo uno de C2P
fold_a_comprobar = "C3_pair_NleD__Mapkapk2"
train_samples = folds[fold_a_comprobar]["train"]

# 3. Filtrar tu DataFrame original (labeled_meta) usando los nombres del train
# (Asumiendo que tu df original se llama labeled_meta)
df_train_audit = labeled_meta[labeled_meta["Sample_name"].isin(train_samples)]

# 4. Agrupar y verificar si algún individuo viola la regla biclase
eff_biclase_check = df_train_audit.groupby("Effector")["Label"].agg(['min', 'max'])
prot_biclase_check = df_train_audit.groupby("Protein")["Label"].agg(['min', 'max'])

# Si el filtro funcionó, 'min' debe ser 0 y 'max' debe ser 1 para TODOS
eff_fallos = eff_biclase_check[(eff_biclase_check['min'] == eff_biclase_check['max'])]
prot_fallos = prot_biclase_check[(prot_biclase_check['min'] == prot_biclase_check['max'])]

print(f"=== AUDITORÍA DEL FOLD: {fold_a_comprobar} ===")
print(f"Efectores individuales en Train que NO son biclase: {len(eff_fallos)}")
print(f"Proteínas individuales en Train que NO son biclase: {len(prot_fallos)}")

if len(eff_fallos) == 0 and len(prot_fallos) == 0:
    print("\n✅ ¡TODO PERFECTO! El filtro iterativo funciona de manera matemática estricta.")
else:
    print("\n❌ Hay algo raro, algunos individuos se han colgado con una sola clase.")

=== AUDITORÍA DEL FOLD: C3_pair_NleD__Mapkapk2 ===
Efectores individuales en Train que NO son biclase: 0
Proteínas individuales en Train que NO son biclase: 0

✅ ¡TODO PERFECTO! El filtro iterativo funciona de manera matemática estricta.


### Cross-validation

In [23]:
# ── Lanzamiento de la Optimización y Entrenamiento MLP ───────────────────────
all_results = test_pooling_transforms_sublevels(
    test_transforms,
    labeled_dir=OUTPUT_DIR,
    splits_dir=SPLITS_DIR_SUBNIVELES,
    metadata_df=labeled_meta
)


######################################################################
#  SUBNIVEL INDIVIDUAL (MLP): C1_C2E
######################################################################

Embedding: single_embeddings (3 poolings)


✅ Éxito: Se cargaron 962 muestras de las 962 del Excel.

  --- MLP | single_embeddings | mean ---


    [Media de los 24 Folds Evaluados]
    ROC AUC: 0.7835 | PR AUC: 0.6167 | Lift: 3.04x
    AUPRG (PR-Gain): 0.5795 | E[FG1] (F1-Gain Esperado): 0.5580
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.4231 | Rec: 0.3829 | Bal.Acc: 0.6449 | PR-Gain: 0.4417 | F1-Gain: 0.3514
      -> Umbral Optimizado [Promedio/Pool: 0.174]:
         Prec: 0.5440 | Rec: 0.5635 | Bal.Acc: 0.6959 | PR-Gain: 0.5854 | F1-Gain: 0.5105
    Tiempo total de cómputo del subnivel: 556.1s

  --- MLP | single_embeddings | max ---


    [Media de los 24 Folds Evaluados]
    ROC AUC: 0.7712 | PR AUC: 0.6355 | Lift: 3.05x
    AUPRG (PR-Gain): 0.6380 | E[FG1] (F1-Gain Esperado): 0.5715
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.5830 | Rec: 0.3977 | Bal.Acc: 0.6498 | PR-Gain: 0.6094 | F1-Gain: 0.4265
      -> Umbral Optimizado [Promedio/Pool: 0.173]:
         Prec: 0.6457 | Rec: 0.5971 | Bal.Acc: 0.7327 | PR-Gain: 0.7481 | F1-Gain: 0.6547
    Tiempo total de cómputo del subnivel: 450.2s

  --- MLP | single_embeddings | min ---


    [Media de los 24 Folds Evaluados]
    ROC AUC: 0.7367 | PR AUC: 0.5870 | Lift: 2.80x
    AUPRG (PR-Gain): 0.5638 | E[FG1] (F1-Gain Esperado): 0.5336
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.4642 | Rec: 0.3977 | Bal.Acc: 0.6495 | PR-Gain: 0.5287 | F1-Gain: 0.3489
      -> Umbral Optimizado [Promedio/Pool: 0.187]:
         Prec: 0.5615 | Rec: 0.5680 | Bal.Acc: 0.7051 | PR-Gain: 0.6499 | F1-Gain: 0.6140
    Tiempo total de cómputo del subnivel: 383.1s

Embedding: pair_embeddings (3 poolings)


...

    [Métricas Consolidadas por Pooling de Bloque]
    ROC AUC: 0.9372 | PR AUC: 0.8336 | Lift: 4.01x
    AUPRG (PR-Gain): 0.9630 | E[FG1] (F1-Gain Esperado): 0.7315
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.6411 | Rec: 0.6819 | Bal.Acc: 0.8093 | PR-Gain: 0.6776 | F1-Gain: 0.6681
      -> Umbral Optimizado [Promedio/Pool: 0.871]:
         Prec: 0.8284 | Rec: 0.7035 | Bal.Acc: 0.8326 | PR-Gain: 0.9456 | F1-Gain: 0.9175
    Tiempo total de cómputo del subnivel: 3009.4s

Embedding: pair_embeddings (3 poolings)


✅ Éxito: Se cargaron 962 muestras de las 962 del Excel.

  --- MLP | pair_embeddings | mean ---


    [Métricas Consolidadas por Pooling de Bloque]
    ROC AUC: 0.9029 | PR AUC: 0.7479 | Lift: 3.60x
    AUPRG (PR-Gain): 0.9221 | E[FG1] (F1-Gain Esperado): 0.7110
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.6373 | Rec: 0.7146 | Bal.Acc: 0.8070 | PR-Gain: 0.6593 | F1-Gain: 0.6871
      -> Umbral Optimizado [Promedio/Pool: 0.564]:
         Prec: 0.6740 | Rec: 0.7688 | Bal.Acc: 0.8356 | PR-Gain: 0.8730 | F1-Gain: 0.8970
    Tiempo total de cómputo del subnivel: 5616.3s

  --- MLP | pair_embeddings | max ---


    [Métricas Consolidadas por Pooling de Bloque]
    ROC AUC: 0.9132 | PR AUC: 0.7597 | Lift: 3.65x
    AUPRG (PR-Gain): 0.9294 | E[FG1] (F1-Gain Esperado): 0.7163
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.6468 | Rec: 0.7069 | Bal.Acc: 0.8094 | PR-Gain: 0.6871 | F1-Gain: 0.6754
      -> Umbral Optimizado [Promedio/Pool: 0.624]:
         Prec: 0.7171 | Rec: 0.7387 | Bal.Acc: 0.8311 | PR-Gain: 0.8964 | F1-Gain: 0.9018
    Tiempo total de cómputo del subnivel: 2262.6s

  --- MLP | pair_embeddings | min ---


    [Métricas Consolidadas por Pooling de Bloque]
    ROC AUC: 0.9218 | PR AUC: 0.8131 | Lift: 3.91x
    AUPRG (PR-Gain): 0.9556 | E[FG1] (F1-Gain Esperado): 0.7278
    [PUNTOS OPERATIVOS CONCRETOS]
      -> Umbral Fijo [0.50]:
         Prec: 0.6927 | Rec: 0.7097 | Bal.Acc: 0.8197 | PR-Gain: 0.7242 | F1-Gain: 0.6965
      -> Umbral Optimizado [Promedio/Pool: 0.396]:
         Prec: 0.7286 | Rec: 0.7688 | Bal.Acc: 0.8468 | PR-Gain: 0.9022 | F1-Gain: 0.9116
    Tiempo total de cómputo del subnivel: 2341.9s

######################################################################
#  SUBNIVEL INDIVIDUAL (MLP): C1_C3
######################################################################

Embedding: single_embeddings (3 poolings)


In [ ]:
# Guardado preventivo del objeto binario completo
import joblib

fichero_salida = "checkpoint_CV_kmeans_PRG_con_EspS_NleK_MLP_subniveles.joblib"
joblib.dump(all_results, fichero_salida)

print(f"\n[OK] ¡Análisis de escenarios completado!")
print(f"[OK] Archivo binario guardado en: '{fichero_salida}'")

In [ ]:
# # Get the joblib back
# import joblib

# all_results = joblib.load("checkpoint_CV_kmeans_PRG_con_EspS_NleK_MLP_subniveles.joblib")

In [ ]:
# Construcción de las tablas Pandas a partir del objeto de entrenamiento
df_results = optresults2table_nested(all_results)
display(df_results)
df_results.to_csv(OUTPUT_RESULTS / "cv_results_all_scenarios_estricto_kmeans_PRG_con_EspS_NleK_MLP.csv", index=False)

In [ ]:
df_folds = fold_detail_table(all_results)

cols = [
    "fold_id", "note", "n_test", "n_test_pos",
    "roc_auc", "pr_auc", "lift", "auprg", "expected_f1g",
    "best_threshold", "precision_50", "recall_50", "f1g_50",
    "precision_opt", "recall_opt", "f1g_opt", "best_hidden_layer_sizes", "best_alpha", "best_lr_init"
]

# Iteramos de forma limpia por los nuevos nombres de tus subniveles + control
for scenario in ["C1", "C1_C2E", "C1_C2P", "C1_C3"]:
    if scenario not in df_results["scenario"].values:
        continue
        
    best_combo = (
        df_results[df_results["scenario"] == scenario]
        .sort_values("auprg", ascending=False)
        .iloc[0]
    )

    sub = df_folds[
        (df_folds["scenario"]       == scenario) &
        (df_folds["representation"] == best_combo["representation"]) &
        (df_folds["pooling"]        == best_combo["pooling"])
    ]

    # --- CONTROL METODOLÓGICO DEL ERROR ESTÁNDAR (SEM) ---
    # Si el escenario usa estrategia de Pooling (acumulado), la varianza fold a fold no existe 
    # para las métricas globales porque fueron calculadas sobre el pool consolidado.
    if scenario in ["C1_C2P", "C1_C3"]:
        error_report = f"(Métrica Consolidada Global por Pooling | E[FG1] = {best_combo['expected_f1_gain']:.4f})"
    else:
        # Para C1 y C1_C2E calculamos la desviación real sobre los folds
        n_folds = len(sub)
        auprg_std = sub['auprg'].std(ddof=1) / np.sqrt(n_folds) if n_folds > 1 else 0.0
        error_report = f"(Media AUPRG = {best_combo['auprg']:.4f} ± {auprg_std:.4f} | E[FG1] = {best_combo['expected_f1_gain']:.4f})"

    print(f"\n{'='*90}")
    print(f"  {scenario} — {best_combo['representation']} | {best_combo['pooling']}")
    print(f"  {error_report}")
    print(f"{'='*90}")
    
    existing_cols = [c for c in cols if c in sub.columns]
    display(sub[existing_cols].reset_index(drop=True))

output_path = OUTPUT_RESULTS / "cv_results_per_fold_estricto_kmeans_PRG_con_EspS_NleK_MLP.csv"
df_folds.to_csv(output_path, index=False)
print(f"\nTabla completa por fold guardada en: {output_path}")

In [ ]:
# 1. Extraemos las matrices del subnivel de control para el embedding ganador
# (Asegúrate de tener definidas estas variables de tu pipeline de carga)
best_emb_type = "single_embeddings"
best_transform = test_transforms[best_emb_type][0] # Tu función de pooling mean/max/min

# Cargamos el bloque limpio usando tu función tradicional
X_full, y_full, _ = load_block_from_csv(
    OUTPUT_DIR, 
    target_df=labeled_meta[labeled_meta['Label'].isin([0,1])], 
    emb_type=best_emb_type, 
    transforms=[best_transform]
)

# Ajustamos el shape si es tridimensional (3, muestras, features) -> (muestras, features)
Xi_check = X_full[:, 0, :] if X_full.ndim == 3 else X_full

# 2. Lanzamos el Sanity Check antes del entrenamiento gordo
sanity_check_random_labels_sublevels(Xi_check, y_full, n_repeats=3)